
# 🧭 Week 3 — Pipelines, Modularity & Debugging with DSPY

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week3_pipelines_and_dspy.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

*How AI engineers move from single prompts to reliable, modular LLM systems.*

---

## 🎯 Learning Objectives

By the end of this week, you should be able to:
1. Explain what a pipeline is and why modularity matters for reliability.  
2. Distinguish between monolithic and multi-step prompt designs.  
3. Configure and run a basic LLM in `dspy` using the OpenAI API.  
4. Build, inspect, and debug simple pipelines step-by-step.  
5. Use `dspy.inspect()` and logging to understand what’s actually happening inside an LLM pipeline.


In [1]:
# @title Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys, platform
sys.path.append('/content/main')
from course_utils import lab3_setup, show_mermaid

lab3_setup()
print(f"✅ Environment ready!")


Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
✅ lab3_setup complete — scientific libraries ready, helper function loaded.
✅ Environment ready!


# 📅 Day 1 — From Prompts to Pipelines


## Section 1 — Lab L2 Recap: What Changed When Temperature Increased?

**Guiding Question:**  
> What did you notice when temperature increased? Was the model more creative, or less reliable?

Discuss:
- Higher temperature → more diverse completions.
- Lower temperature → more deterministic and stable outputs.



## Section 2 — Motivating Example: Why Modularity?

**Question:**  
> Why can’t we just write one giant prompt and call it a day?

We'll explore examples of LLM success and failure:
- **Success:** multi-step reasoning task (summarize → critique → rewrite).
- **Failure:** same task with one unstructured prompt.

**Reflection:**  
> What might have gone wrong?  
> How could you design the system to isolate each step?



## Section 3 — Concept: What Is a Pipeline?

A **pipeline** is a sequence of processing steps that move data forward.  
Each stage transforms or filters information before handing it off.

**Mathematical intuition:**

$$
P(Y|X) = \prod_{i=1}^{n} P(Y_i | Y_{<i}, X)
$$



In [ ]:
# @title summarization pipeline
show_mermaid("""graph TD
  I["Input"] --> E["Extract"]
  E --> S["Simplify"]
  S --> Z["Summarize"]
  Z --> O["Output"]
""")

In [ ]:
# @title Updated LLM Diagram
show_mermaid("""
graph TD
    %% === USER INPUT ===
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    %% === PIPELINE STRUCTURE ===
    subgraph Pipeline Modules
    IH --> M1["Module 1: Extraction<br/>• Pull key facts or entities"]:::module
    M1 --> M2["Module 2: Reasoning<br/>• Infer relationships<br/>• Compute answers"]:::module
    M2 --> M3["Module 3: Summarization<br/>• Convert structured data<br/>• Produce final text output"]:::module
    end

    %% === MODEL & DATA CONNECTIONS ===
    subgraph Core LLMs
    M1 --> L1["LLM A<br/>Specialized extractor"]:::model
    M2 --> L2["LLM B<br/>Reasoner / Planner"]:::model
    M3 --> L3["LLM C<br/>Writer / Stylist"]:::model
    end

    subgraph Model Training
    D["📚 Training Data Distribution<br/>• Text corpus<br/>• Domain knowledge<br/>• Bias sources"]:::data --> L1
    D --> L2
    D --> L3
    end

    %% === OUTPUT & MONITORING ===
    subgraph Output & Monitoring
    M3 --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Guardrails"]:::output
    OP --> O["🟢 Final Output"]:::output
    O --> LM["Logging & Monitoring<br/>• Per-module metrics<br/>• Failure localization<br/>• Drift detection"]:::monitor
    end

    %% === STYLES ===
    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef module fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef data fill:#fde2e4,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
""")



## Section 4 — Mini DSPY Preview: The Simplest Predictor

**Guiding Question:**  
> What does it mean to describe a prompt as a *function*?


In [ ]:
# @title configure dspy to use openai
import dspy
import os
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [ ]:
# @title simple dspy function
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")
print(result.answer)


The capital of France is Paris.


In [ ]:
# @title inspect prompt
dspy.inspect_history()






[2025-12-28T15:34:41.495452]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What is the capital of France?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
The capital of France is Paris.

[[ ## completed ## ]]








## Section 5 — Debugging by Decomposition

**Guiding Question:**  
> What happens when a single step in a multi-step system fails?


In [ ]:
# @title sample failures
import random

steps = ["extract", "simplify", "summarize"]
for s in steps:
    success = random.random() > 0.3
    print(f"{s}: {'✅ success' if success else '❌ failure'}")


extract: ✅ success
simplify: ✅ success
summarize: ❌ failure



## Section 6 — Activity: Sketch a Real AI Pipeline

In pairs, pick an AI system (e.g., ChatGPT, Grammarly, Copilot).  
Sketch its internal pipeline (3–4 boxes max):  
*Input Handling → Intent Detection → LLM → Postprocessing*



## Section 7 — 5-Minute Concept Quiz

1. Why is modularity useful in AI pipelines?  
2. What is “failure localization”?  
3. Why might debugging be harder in a single giant prompt?  
4. How does `dspy` help with modularity?  
5. What does `dspy.inspect()` show?



## Section 8 — Unifying Diagram v2



In [ ]:
# @title pipeline

show_mermaid(
"""
graph TD
  U["User Input"] --> IH["Input Handling"]
  IH --> S1["Step 1: Extract Info"]
  S1 --> S2["Step 2: Simplify"]
  S2 --> LLM["Core LLM"]
  LLM --> OP["Output Processing"]
  OP --> M["Monitoring"]
"""
)


<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 1)</b></summary>

- Timing: 10 + 10 + 15 + 15 + 15 + 5 + 5  
- Use `dspy.inspect()` live to reveal prompts.  
- Encourage students to draw diagrams and share debugging analogies.  
- Wrap with a quiz and transition: “Next class, we’ll *build* pipelines in DSPY.”

</details>


# 📅 Day 2 — Building and Debugging Pipelines with DSPY


## Section 2 — Typed Predictors


In [ ]:
# @title adding types
sentiment = dspy.Predict("sentence -> sentiment: bool")
result = sentiment(sentence="I love debugging pipelines!")
print("Sentiment:", result.sentiment)


Sentiment: True


In [ ]:
# @title multiple outputs
qa = dspy.Predict("question -> reasoning, answer")
resp = qa(question="Why do leaves change color in autumn?")
print("Reasoning:", resp.reasoning)
print("Answer:", resp.answer)


Reasoning: Leaves change color in autumn primarily due to the breakdown of chlorophyll, the green pigment responsible for photosynthesis. As days shorten and temperatures drop, chlorophyll production slows and eventually ceases. This allows other pigments, such as carotenoids (which produce yellow and orange hues) and anthocyanins (which produce red and purple colors), to become more visible. Additionally, environmental factors like temperature, sunlight, and soil moisture can influence the intensity and timing of these color changes.
Answer: Leaves change color in autumn due to the breakdown of chlorophyll, revealing other pigments like carotenoids and anthocyanins as daylight decreases and temperatures drop.


In [ ]:
# @title summarization pipeline
extract = dspy.Predict("article -> key_points")
simplify = dspy.Predict("key_points -> summary")
tag = dspy.Predict("summary -> tags: list[str]")

def summarize_pipeline(article):
    key_points = extract(article=article).key_points
    summary = simplify(key_points=key_points).summary
    tags = tag(summary=summary).tags
    return summary, tags

article = "Photosynthesis converts light energy into chemical energy..."
print(summarize_pipeline(article))


('Photosynthesis is the mechanism through which light energy is transformed into chemical energy, mainly occurring in plants, algae, and certain bacteria. The key outcome of this process is glucose, which provides energy for the organism, while oxygen is released as a byproduct. Photosynthesis takes place in chloroplasts, utilizing chlorophyll to harness light energy.', ['photosynthesis', 'light energy', 'chemical energy', 'plants', 'algae', 'bacteria', 'glucose', 'oxygen', 'chloroplasts', 'chlorophyll'])


In [ ]:
# @title Using try/except blocks to safely execute pipeline steps
def safe_run(predictor, **kwargs):
    try:
        result = predictor(**kwargs)
        if not result:
            raise ValueError("Empty output")
        return result
    except Exception as e:
        print(f"⚠️ Error in {predictor}: {e}")
        return None

result = safe_run(extract, text="...")
result


2025/12/28 15:44:24 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: []. Missing: ['article'].


Prediction(
    key_points='{key_points}'
)

In [ ]:
# @title sample trace
import pandas as pd

trace = [
    {"step": "extract", "status": "✅", "details": "3 key points"},
    {"step": "simplify", "status": "✅", "details": "Readable summary"},
    {"step": "tag", "status": "⚠️", "details": "Low confidence tags"}
]

pd.DataFrame(trace)


,step,status,details
0,extract,✅,3 key points
1,simplify,✅,Readable summary
2,tag,⚠️,Low confidence tags


In [ ]:
# @title Using Classes with dspy
class Summarizer(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict("article -> key_points")
        self.simplify = dspy.Predict("key_points -> summary")
    def forward(self, article):
        return self.simplify(key_points=self.extract(article=article).key_points).summary

module = Summarizer()
print(module("The sun provides energy for plants..."))


The sun serves as the primary energy source for plants, enabling them to perform photosynthesis, through which they produce food. Sunlight is crucial for the growth and development of plants.


In [ ]:
dspy.inspect_history(n=2)





[2025-12-28T15:43:06.361046]

System message:

Your input fields are:
1. `article` (str):
Your output fields are:
1. `key_points` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## article ## ]]
{article}

[[ ## key_points ## ]]
{key_points}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `article`, produce the fields `key_points`.


User message:

[[ ## article ## ]]
The sun provides energy for plants...

Respond with the corresponding output fields, starting with the field `[[ ## key_points ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## key_points ## ]]
- The sun is a primary energy source for plants.
- Plants utilize sunlight during photosynthesis to produce food.
- Sunlight is essential for plant growth and development.

[[ ## completed ## ]]





[2025-12-28T15:43:06.438919]

System message:

Your input fields are:
1. `key_poin

In [ ]:
# @title logging steps
logs = []
article = "AI systems are more interpretable when modular."
summary = module(article)
logs.append({"input": article, "output": summary})

import pandas as pd
pd.DataFrame(logs)


,input,output
0,AI systems are more interpretable when modular.,Modular AI systems improve the interpretabilit...



## Section 8 — Wrap-Up Discussion

- Compare: Monolithic vs modular prompts.  
- Review: `dspy.inspect()`, error handling, logging, and modularization.  
- Preview: Next week—**Embeddings** and **semantic retrieval**.



<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 2)</b></summary>

- Timing: 10 + 10 + 15 + 20 + 10 + 5 + 5  
- Have students run `dspy.inspect()` and `safe_run()` interactively.  
- Encourage experimentation: intentionally break steps and observe errors.  
- Close by linking debugging practices to system reliability.

</details>
